***Assignment - Loan Approval Prediction:***

Using the Loan Approval dataset, create an end-to-end workflow for predicting loan approval. Your workflow should include:

- Data loading and exploration
- Data preprocessing (handling missing values, encoding categorical variables, feature scaling)
- Feature selection
- Model training (using logistic regression and KNN)
- Model evaluation (using accuracy, precision, recall, F1-score and ROC AUC score)

```python
import pandas as pd
import numpy as np

# Load the dataset
loan_data = pd.read_csv('https://raw.githubusercontent.com/prasertcbs/basic-dataset/refs/heads/master/Loan-Approval-Prediction.csv')

# Split features and target
X = loan_data.drop('Loan_Status', axis=1)
y = loan_data['Loan_Status']
```


In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

# Model Training
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

# 2. Load the dataset
loan_data = pd.read_csv(
    'https://raw.githubusercontent.com/prasertcbs/basic-dataset/refs/heads/master/Loan-Approval-Prediction.csv'
)

# Check the shape of the dataset
print("Dataset shape:")
print(loan_data.shape)

# Display first 5 rows
print(loan_data.head())

# Check data types
print("\nData types:")
print(loan_data.dtypes)

# Check missing values
print("\nMissing values:")
print(loan_data.isnull().sum())

# Check target distribution
print("\nLoan_Status value counts:")
print(loan_data['Loan_Status'].value_counts())

Dataset shape:
(614, 13)
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural          

In [15]:
# 3. Split features and target
X = loan_data.drop('Loan_Status', axis=1)
y = loan_data['Loan_Status']

# Encode target variable: Y = 1, N = 0
y = y.map({'Y': 1, 'N': 0})


# 4. Feature selection
# Drop Loan_ID because it is only an identifier and does not help in prediction
X = X.drop('Loan_ID', axis=1)

print("\nSelected features:")
print(X.columns)


Selected features:
Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')


In [16]:
# 5. Define numerical and categorical columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


Numerical features:
['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

Categorical features:
['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']


In [17]:
# 6. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape :", X_test.shape)


Training set shape: (491, 11)
Testing set shape : (123, 11)


In [ ]:
# 7. Logistic Regression workflow
# Numerical: median imputation + StandardScaler
# Categorical: most frequent imputation + OneHotEncoder


# Numerical preprocessing for Logistic Regression
numerical_transformer_lr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # fill missing numerical values with median
    ('scaler', StandardScaler())                     # standardize numerical values
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),   # fill missing categorical values with mode
    ('encoder', OneHotEncoder(handle_unknown='ignore'))     # one-hot encode categorical variables
])

# Combine numerical and categorical preprocessing
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer_lr, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Create full pipeline for Logistic Regression
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('model', LogisticRegression(max_iter=1000))
])

# Train the Logistic Regression model
pipeline_lr.fit(X_train, y_train)

# Predict class labels
y_pred_lr = pipeline_lr.predict(X_test)

# Predict probabilities for ROC AUC
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]

# Evaluate Logistic Regression
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_prob_lr)
roc_auc_lr = auc(fpr_lr, tpr_lr)

print("\n===== Logistic Regression Results =====")
print(f"Accuracy : {accuracy_lr:.2f}")
print(f"Precision: {precision_lr:.2f}")
print(f"Recall   : {recall_lr:.2f}")
print(f"F1 Score : {f1_lr:.2f}")
print(f"ROC AUC  : {roc_auc_lr:.2f}")


===== Logistic Regression Results =====
Accuracy : 0.84
Precision: 0.83
Recall   : 0.98
F1 Score : 0.90
ROC AUC  : 0.83


In [19]:

# 8. KNN workflow
# Numerical: median imputation + MinMaxScaler
# Categorical: most frequent imputation + OneHotEncoder

# Numerical preprocessing for KNN
numerical_transformer_knn = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # fill missing numerical values with median
    ('scaler', MinMaxScaler())                       # scale numerical values to range 0 to 1
])

# Combine numerical and categorical preprocessing
preprocessor_knn = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer_knn, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Create full pipeline for KNN
pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor_knn),
    ('model', KNeighborsClassifier())
])

# Train the KNN model
pipeline_knn.fit(X_train, y_train)

# Predict class labels
y_pred_knn = pipeline_knn.predict(X_test)

# Predict probabilities for ROC AUC
y_prob_knn = pipeline_knn.predict_proba(X_test)[:, 1]

# Evaluate KNN
accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

fpr_knn, tpr_knn, thresholds_knn = roc_curve(y_test, y_prob_knn)
roc_auc_knn = auc(fpr_knn, tpr_knn)

print("\n===== KNN Results =====")
print(f"Accuracy : {accuracy_knn:.2f}")
print(f"Precision: {precision_knn:.2f}")
print(f"Recall   : {recall_knn:.2f}")
print(f"F1 Score : {f1_knn:.2f}")
print(f"ROC AUC  : {roc_auc_knn:.2f}")


===== KNN Results =====
Accuracy : 0.75
Precision: 0.79
Recall   : 0.89
F1 Score : 0.84
ROC AUC  : 0.71


In [20]:

# 9. Compare both models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'KNN'],
    'Accuracy': [accuracy_lr, accuracy_knn],
    'Precision': [precision_lr, precision_knn],
    'Recall': [recall_lr, recall_knn],
    'F1 Score': [f1_lr, f1_knn],
    'ROC AUC': [roc_auc_lr, roc_auc_knn]
})

print("\n===== Model Comparison =====")
print(results)


===== Model Comparison =====
                 Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
0  Logistic Regression  0.837398   0.830189  0.977778  0.897959  0.826263
1                  KNN  0.747967   0.792079  0.888889  0.837696  0.713300
